# EDA - Goal 2: Analisis Kerentanan Ekonomi Wilayah Koperasi
**Wilayah Studi:** Kabupaten Banyumas, Jawa Tengah  
**Tujuan:** Menganalisis kerentanan ekonomi wilayah koperasi, mengidentifikasi koperasi dengan risiko tinggi, dan zona prioritas intervensi pemerintah

## 1. Install & Import Library

In [ ]:
!pip install geopandas folium mapclassify matplotlib seaborn scipy -q

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.spatial import cKDTree
import folium
from folium.plugins import HeatMap, MarkerCluster
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

print('Library siap!')

## 2. Load Semua Data

In [ ]:
import os

DATA_DIR = r"D:\semester 6\Big Data IOT\koperasi\data"

# Load semua data
koperasi    = pd.read_csv(os.path.join(DATA_DIR, "KDMP Banyumas.csv"))
pemukiman   = gpd.read_file(os.path.join(DATA_DIR, "pemukiman.geojson"))
faskes      = gpd.read_file(os.path.join(DATA_DIR, "faskes.geojson"))
pendidikan  = gpd.read_file(os.path.join(DATA_DIR, "pendidikan.geojson"))
jalan       = gpd.read_file(os.path.join(DATA_DIR, "jaringan jalan.geojson"))
kepadatan   = gpd.read_file(os.path.join(DATA_DIR, "kepadatan penduduk.geojson"))
kemiskinan  = pd.read_csv(os.path.join(DATA_DIR, "kemiskinan.csv"))
pengangguran= pd.read_csv(os.path.join(DATA_DIR, "pengangguran.csv"))

print("Semua data berhasil di-load!")
print(f"  Koperasi      : {len(koperasi)} baris")
print(f"  Pemukiman     : {len(pemukiman)} fitur")
print(f"  Faskes        : {len(faskes)} fitur")
print(f"  Pendidikan    : {len(pendidikan)} fitur")
print(f"  Jalan         : {len(jalan)} fitur")
print(f"  Kepadatan     : {len(kepadatan)} fitur")
print(f"  Kemiskinan    : {len(kemiskinan)} baris")
print(f"  Pengangguran  : {len(pengangguran)} baris")

---
## 3. Preprocessing Data Koperasi
Parse koordinat dan konversi ke GeoDataFrame

In [ ]:
print("Kolom koperasi:", koperasi.columns.tolist())
print(koperasi.head(3))

In [ ]:
# Parse koordinat dari kolom 'Koordinat' yang formatnya "-7.xxx, 109.xxx"
koperasi[['lat', 'lon']] = koperasi['Koordinat'].str.split(',', expand=True).astype(float)

# Konversi ke GeoDataFrame
gdf_koperasi = gpd.GeoDataFrame(
    koperasi,
    geometry=gpd.points_from_xy(koperasi['lon'], koperasi['lat']),
    crs='EPSG:4326'
)

print(f"Total koperasi: {len(gdf_koperasi)}")
print(f"Kecamatan unik: {gdf_koperasi['Kecamatan'].nunique()}")
print(f"Progres pembangunan unik: {sorted(gdf_koperasi['Kategori Pembangunan'].unique())}")

---
## 4. EDA Per Variabel
### 4.1 Koperasi KDMP

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribusi progres pembangunan
kategori_count = gdf_koperasi['Kategori Pembangunan'].value_counts()
axes[0].bar(kategori_count.index, kategori_count.values, color=['#2ecc71','#f39c12','#e74c3c','#3498db'])
axes[0].set_title('Distribusi Kategori Pembangunan Koperasi')
axes[0].set_xlabel('Kategori')
axes[0].set_ylabel('Jumlah Koperasi')
axes[0].tick_params(axis='x', rotation=30)

# Top 10 kecamatan dengan koperasi terbanyak
top_kec = gdf_koperasi['Kecamatan'].value_counts().head(10)
axes[1].barh(top_kec.index, top_kec.values, color='#3498db')
axes[1].set_title('Top 10 Kecamatan - Jumlah Koperasi')
axes[1].set_xlabel('Jumlah Koperasi')

# Histogram progres pembangunan (%)
axes[2].hist(gdf_koperasi['Progres Pembangunan (%)'], bins=15, color='#9b59b6', edgecolor='white')
axes[2].set_title('Distribusi Progres Pembangunan (%)')
axes[2].set_xlabel('Progres (%)')
axes[2].set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('eda_koperasi.png', bbox_inches='tight')
plt.show()

print(gdf_koperasi[['Progres Pembangunan (%)']].describe())

### 4.2 Pemukiman (Building Footprint)

In [ ]:
print("=== INFO PEMUKIMAN ===")
print(f"Jumlah bangunan : {len(pemukiman):,}")
print(f"CRS             : {pemukiman.crs}")
print(f"Kolom           : {pemukiman.columns.tolist()}")
print(f"Tipe geometri   : {pemukiman.geometry.geom_type.value_counts().to_dict()}")
print(f"Missing values  :\n{pemukiman.isnull().sum()}")

In [ ]:
# Hitung luas bangunan jika ada kolom area_m2, jika tidak hitung ulang
if 'area_m2' not in pemukiman.columns:
    pemukiman_utm = pemukiman.to_crs(epsg=32749)
    pemukiman['area_m2'] = pemukiman_utm.geometry.area

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribusi luas bangunan (filter outlier >500m2 untuk visualisasi)
luas_filtered = pemukiman['area_m2'][pemukiman['area_m2'] < 500]
axes[0].hist(luas_filtered, bins=40, color='#e67e22', edgecolor='white')
axes[0].set_title('Distribusi Luas Bangunan (< 500 m²)')
axes[0].set_xlabel('Luas (m²)')
axes[0].set_ylabel('Frekuensi')

# Statistik deskriptif
stats = pemukiman['area_m2'].describe()
axes[1].barh(stats.index, stats.values, color='#e67e22')
axes[1].set_title('Statistik Luas Bangunan (m²)')
axes[1].set_xlabel('Nilai')

# Tipe bangunan jika ada kolomnya
if 'building' in pemukiman.columns:
    top_type = pemukiman['building'].value_counts().head(10)
    axes[2].barh(top_type.index.astype(str), top_type.values, color='#e67e22')
    axes[2].set_title('Top 10 Tipe Bangunan')
    axes[2].set_xlabel('Jumlah')
else:
    axes[2].text(0.5, 0.5, 'Kolom tipe bangunan\ntidak tersedia', 
                ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Tipe Bangunan')

plt.tight_layout()
plt.savefig('eda_pemukiman.png', bbox_inches='tight')
plt.show()

print(pemukiman['area_m2'].describe())

### 4.3 Fasilitas Kesehatan

In [ ]:
print("=== INFO FASKES ===")
print(f"Jumlah fitur : {len(faskes):,}")
print(f"CRS          : {faskes.crs}")
print(f"Kolom        : {faskes.columns.tolist()}")
print(f"Missing values:\n{faskes.isnull().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cari kolom tipe faskes
type_col = None
for c in ['amenity', 'healthcare', 'type', 'jenis']:
    if c in faskes.columns:
        type_col = c
        break

if type_col:
    vc = faskes[type_col].value_counts()
    axes[0].bar(vc.index.astype(str), vc.values, color='#e74c3c')
    axes[0].set_title(f'Distribusi Jenis Fasilitas Kesehatan ({type_col})')
    axes[0].set_xlabel('Jenis')
    axes[0].set_ylabel('Jumlah')
    axes[0].tick_params(axis='x', rotation=45)
else:
    axes[0].text(0.5, 0.5, 'Kolom tipe faskes\ntidak ditemukan',
                ha='center', va='center', transform=axes[0].transAxes)

# Cari kolom nama kecamatan
kec_col = None
for c in ['kecamatan', 'addr:city', 'district', 'Kecamatan']:
    if c in faskes.columns:
        kec_col = c
        break

if kec_col:
    top_kec = faskes[kec_col].value_counts().head(10)
    axes[1].barh(top_kec.index.astype(str), top_kec.values, color='#e74c3c')
    axes[1].set_title('Top 10 Kecamatan - Jumlah Faskes')
    axes[1].set_xlabel('Jumlah')
else:
    # distribusi koordinat lat sebagai alternatif
    faskes_pt = faskes.copy()
    if faskes_pt.geometry.geom_type.isin(['Polygon','MultiPolygon']).any():
        faskes_pt['geometry'] = faskes_pt.geometry.centroid
    axes[1].hist(faskes_pt.geometry.y, bins=20, color='#e74c3c', edgecolor='white')
    axes[1].set_title('Distribusi Latitude Faskes')
    axes[1].set_xlabel('Latitude')

plt.tight_layout()
plt.savefig('eda_faskes.png', bbox_inches='tight')
plt.show()

### 4.4 Fasilitas Pendidikan

In [ ]:
print("=== INFO PENDIDIKAN ===")
print(f"Jumlah fitur : {len(pendidikan):,}")
print(f"CRS          : {pendidikan.crs}")
print(f"Kolom        : {pendidikan.columns.tolist()}")
print(f"Missing values:\n{pendidikan.isnull().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

type_col_edu = None
for c in ['amenity', 'type', 'jenis', 'school:type']:
    if c in pendidikan.columns:
        type_col_edu = c
        break

if type_col_edu:
    vc = pendidikan[type_col_edu].value_counts()
    axes[0].bar(vc.index.astype(str), vc.values, color='#3498db')
    axes[0].set_title(f'Distribusi Jenis Fasilitas Pendidikan ({type_col_edu})')
    axes[0].set_xlabel('Jenis')
    axes[0].set_ylabel('Jumlah')
    axes[0].tick_params(axis='x', rotation=45)
else:
    axes[0].text(0.5, 0.5, 'Kolom tipe pendidikan\ntidak ditemukan',
                ha='center', va='center', transform=axes[0].transAxes)

# Cari kolom nama kecamatan
kec_col_edu = None
for c in ['kecamatan', 'addr:city', 'district', 'Kecamatan']:
    if c in pendidikan.columns:
        kec_col_edu = c
        break

if kec_col_edu:
    top = pendidikan[kec_col_edu].value_counts().head(10)
    axes[1].barh(top.index.astype(str), top.values, color='#3498db')
    axes[1].set_title('Top 10 Kecamatan - Jumlah Fasilitas Pendidikan')
    axes[1].set_xlabel('Jumlah')
else:
    pendidikan_pt = pendidikan.copy()
    if pendidikan_pt.geometry.geom_type.isin(['Polygon','MultiPolygon']).any():
        pendidikan_pt['geometry'] = pendidikan_pt.geometry.centroid
    axes[1].hist(pendidikan_pt.geometry.y, bins=20, color='#3498db', edgecolor='white')
    axes[1].set_title('Distribusi Latitude Fasilitas Pendidikan')
    axes[1].set_xlabel('Latitude')

plt.tight_layout()
plt.savefig('eda_pendidikan.png', bbox_inches='tight')
plt.show()

### 4.5 Jaringan Jalan

In [ ]:
print("=== INFO JARINGAN JALAN ===")
print(f"Jumlah fitur : {len(jalan):,}")
print(f"CRS          : {jalan.crs}")
print(f"Kolom        : {jalan.columns.tolist()}")
print(f"Missing values:\n{jalan.isnull().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Tipe jalan
highway_col = None
for c in ['highway', 'type', 'road_type', 'jenis_jalan']:
    if c in jalan.columns:
        highway_col = c
        break

if highway_col:
    vc = jalan[highway_col].explode().value_counts().head(12)
    axes[0].barh(vc.index.astype(str), vc.values, color='#7f8c8d')
    axes[0].set_title(f'Distribusi Tipe Jalan ({highway_col})')
    axes[0].set_xlabel('Jumlah Segmen')
else:
    axes[0].text(0.5, 0.5, 'Kolom tipe jalan\ntidak ditemukan',
                ha='center', va='center', transform=axes[0].transAxes)

# Panjang jalan jika ada
if 'length' in jalan.columns:
    jalan_filtered = jalan[jalan['length'] < jalan['length'].quantile(0.95)]
    axes[1].hist(jalan_filtered['length'], bins=40, color='#7f8c8d', edgecolor='white')
    axes[1].set_title('Distribusi Panjang Segmen Jalan')
    axes[1].set_xlabel('Panjang (meter)')
    axes[1].set_ylabel('Frekuensi')
else:
    # Hitung panjang dari geometry
    jalan_utm = jalan.to_crs(epsg=32749)
    jalan['length_m'] = jalan_utm.geometry.length
    jalan_filtered = jalan[jalan['length_m'] < jalan['length_m'].quantile(0.95)]
    axes[1].hist(jalan_filtered['length_m'], bins=40, color='#7f8c8d', edgecolor='white')
    axes[1].set_title('Distribusi Panjang Segmen Jalan (dihitung)')
    axes[1].set_xlabel('Panjang (meter)')
    axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('eda_jalan.png', bbox_inches='tight')
plt.show()

### 4.6 Kepadatan Penduduk

In [ ]:
print("=== INFO KEPADATAN PENDUDUK ===")
print(f"Jumlah fitur : {len(kepadatan):,}")
print(f"CRS          : {kepadatan.crs}")
print(f"Kolom        : {kepadatan.columns.tolist()}")
print(kepadatan.head(3))
print(f"Missing values:\n{kepadatan.isnull().sum()}")

In [ ]:
# Cari kolom kepadatan
density_col = None
for c in kepadatan.columns:
    if any(k in c.lower() for k in ['density', 'kepadatan', 'penduduk', 'pop', 'count']):
        density_col = c
        break

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if density_col:
    axes[0].hist(kepadatan[density_col].dropna(), bins=30, color='#1abc9c', edgecolor='white')
    axes[0].set_title(f'Distribusi Kepadatan Penduduk ({density_col})')
    axes[0].set_xlabel('Kepadatan')
    axes[0].set_ylabel('Frekuensi')

    axes[1].boxplot(kepadatan[density_col].dropna(), vert=False, patch_artist=True,
                   boxprops=dict(facecolor='#1abc9c', alpha=0.7))
    axes[1].set_title(f'Boxplot Kepadatan Penduduk ({density_col})')
    axes[1].set_xlabel('Kepadatan')

    print(f"\nStatistik {density_col}:")
    print(kepadatan[density_col].describe())
else:
    print("Kolom kepadatan tidak ditemukan secara otomatis.")
    print("Kolom yang tersedia:", kepadatan.columns.tolist())
    axes[0].text(0.5, 0.5, 'Sesuaikan nama kolom\nkepadatan di kode ini',
                ha='center', va='center', transform=axes[0].transAxes)

plt.tight_layout()
plt.savefig('eda_kepadatan.png', bbox_inches='tight')
plt.show()

### 4.7 Kemiskinan

In [ ]:
print("=== INFO KEMISKINAN ===")
print(kemiskinan.head())
print(f"\nKolom: {kemiskinan.columns.tolist()}")
print(f"Missing values:\n{kemiskinan.isnull().sum()}")

In [ ]:
# Cari kolom kemiskinan dan kecamatan
miskin_col = None
for c in kemiskinan.columns:
    if any(k in c.lower() for k in ['miskin', 'kemiskinan', 'poverty', 'pct', 'persen', '%']):
        miskin_col = c
        break

kec_col_mis = None
for c in kemiskinan.columns:
    if any(k in c.lower() for k in ['kecamatan', 'district', 'wilayah', 'nama']):
        kec_col_mis = c
        break

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if miskin_col and kec_col_mis:
    df_sorted = kemiskinan.sort_values(miskin_col, ascending=False)

    # Bar chart per kecamatan
    axes[0].barh(df_sorted[kec_col_mis].astype(str), df_sorted[miskin_col],
                color='#e74c3c', alpha=0.8)
    axes[0].axvline(df_sorted[miskin_col].mean(), color='black', linestyle='--',
                   label=f'Rata-rata: {df_sorted[miskin_col].mean():.2f}')
    axes[0].set_title('Tingkat Kemiskinan per Kecamatan (%)')
    axes[0].set_xlabel('Persentase Kemiskinan')
    axes[0].legend()

    # Distribusi
    axes[1].hist(kemiskinan[miskin_col].dropna(), bins=15, color='#e74c3c', edgecolor='white')
    axes[1].set_title('Distribusi Tingkat Kemiskinan')
    axes[1].set_xlabel('Persentase Kemiskinan (%)')
    axes[1].set_ylabel('Jumlah Kecamatan')

    print(f"\nStatistik {miskin_col}:")
    print(kemiskinan[miskin_col].describe())
else:
    print("Kolom kemiskinan/kecamatan tidak terdeteksi otomatis.")
    print("Kolom tersedia:", kemiskinan.columns.tolist())

plt.tight_layout()
plt.savefig('eda_kemiskinan.png', bbox_inches='tight')
plt.show()

### 4.8 Pengangguran

In [ ]:
print("=== INFO PENGANGGURAN ===")
print(pengangguran.head())
print(f"\nKolom: {pengangguran.columns.tolist()}")
print(f"Missing values:\n{pengangguran.isnull().sum()}")

In [ ]:
# Cari kolom pengangguran dan kecamatan
tpt_col = None
for c in pengangguran.columns:
    if any(k in c.lower() for k in ['pengangguran', 'tpt', 'unemployment', 'kerja', 'persen', '%']):
        tpt_col = c
        break

kec_col_tpt = None
for c in pengangguran.columns:
    if any(k in c.lower() for k in ['kecamatan', 'district', 'wilayah', 'nama']):
        kec_col_tpt = c
        break

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if tpt_col and kec_col_tpt:
    df_sorted = pengangguran.sort_values(tpt_col, ascending=False)

    axes[0].barh(df_sorted[kec_col_tpt].astype(str), df_sorted[tpt_col],
                color='#f39c12', alpha=0.8)
    axes[0].axvline(df_sorted[tpt_col].mean(), color='black', linestyle='--',
                   label=f'Rata-rata: {df_sorted[tpt_col].mean():.2f}')
    axes[0].set_title('Tingkat Pengangguran per Kecamatan (%)')
    axes[0].set_xlabel('Persentase Pengangguran')
    axes[0].legend()

    axes[1].hist(pengangguran[tpt_col].dropna(), bins=15, color='#f39c12', edgecolor='white')
    axes[1].set_title('Distribusi Tingkat Pengangguran')
    axes[1].set_xlabel('Persentase Pengangguran (%)')
    axes[1].set_ylabel('Jumlah Kecamatan')

    print(f"\nStatistik {tpt_col}:")
    print(pengangguran[tpt_col].describe())
else:
    print("Kolom pengangguran/kecamatan tidak terdeteksi otomatis.")
    print("Kolom tersedia:", pengangguran.columns.tolist())

plt.tight_layout()
plt.savefig('eda_pengangguran.png', bbox_inches='tight')
plt.show()

---
## 5. EDA Gabungan Antar Variabel
### 5.1 Peta Semua Variabel Sekaligus

In [ ]:
# Pastikan semua GeoDataFrame CRS sama
target_crs = 'EPSG:4326'
pemukiman   = pemukiman.to_crs(target_crs)
faskes      = faskes.to_crs(target_crs)
pendidikan  = pendidikan.to_crs(target_crs)
jalan       = jalan.to_crs(target_crs)
kepadatan   = kepadatan.to_crs(target_crs)

# Buat centroid untuk faskes & pendidikan jika masih polygon
faskes_pt = faskes.copy()
if faskes_pt.geometry.geom_type.isin(['Polygon','MultiPolygon']).any():
    faskes_pt['geometry'] = faskes_pt.geometry.centroid

pendidikan_pt = pendidikan.copy()
if pendidikan_pt.geometry.geom_type.isin(['Polygon','MultiPolygon']).any():
    pendidikan_pt['geometry'] = pendidikan_pt.geometry.centroid

fig, ax = plt.subplots(figsize=(14, 14))

pemukiman.plot(ax=ax, color='#f39c12', alpha=0.2, linewidth=0, label='Bangunan/Pemukiman')
jalan.plot(ax=ax, color='#7f8c8d', linewidth=0.3, alpha=0.6, label='Jalan')
faskes_pt.plot(ax=ax, color='#e74c3c', markersize=20, alpha=0.9, label='Faskes', zorder=5)
pendidikan_pt.plot(ax=ax, color='#3498db', markersize=15, alpha=0.9, label='Pendidikan', zorder=5)
gdf_koperasi.plot(ax=ax, color='#2ecc71', markersize=40, marker='*', alpha=1.0,
                  label='Koperasi KDMP', zorder=10)

plt.title('Peta Semua Variabel - Kabupaten Banyumas', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.savefig('peta_semua_variabel.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Hitung Jarak Tiap Koperasi ke Fasilitas Terdekat (Euclidean Distance)

In [ ]:
# Proyeksikan ke UTM untuk hitung jarak dalam meter
kop_utm      = gdf_koperasi.to_crs(epsg=32749)
faskes_utm   = faskes_pt.to_crs(epsg=32749)
pend_utm     = pendidikan_pt.to_crs(epsg=32749)
jalan_utm    = jalan.to_crs(epsg=32749)
pemuk_utm    = pemukiman.to_crs(epsg=32749)

def nearest_distance(gdf_from, gdf_to):
    """Hitung jarak Euclidean ke fitur terdekat (dalam meter)"""
    coords_to = np.array([(geom.centroid.x, geom.centroid.y) 
                          if geom.geom_type != 'Point' 
                          else (geom.x, geom.y) 
                          for geom in gdf_to.geometry])
    coords_from = np.array([(geom.x, geom.y) for geom in gdf_from.geometry])
    tree = cKDTree(coords_to)
    dist, _ = tree.query(coords_from, k=1)
    return dist

gdf_koperasi['jarak_faskes_m']    = nearest_distance(kop_utm, faskes_utm)
gdf_koperasi['jarak_pendidikan_m']= nearest_distance(kop_utm, pend_utm)
gdf_koperasi['jarak_jalan_m']     = nearest_distance(kop_utm, jalan_utm)
gdf_koperasi['jarak_pemukiman_m'] = nearest_distance(kop_utm, pemuk_utm)

print("Statistik jarak koperasi ke fasilitas (meter):")
print(gdf_koperasi[['jarak_faskes_m','jarak_pendidikan_m',
                     'jarak_jalan_m','jarak_pemukiman_m']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

cols_jarak = [
    ('jarak_faskes_m',     'Jarak ke Faskes Terdekat',     '#e74c3c'),
    ('jarak_pendidikan_m', 'Jarak ke Pendidikan Terdekat', '#3498db'),
    ('jarak_jalan_m',      'Jarak ke Jalan Terdekat',      '#7f8c8d'),
    ('jarak_pemukiman_m',  'Jarak ke Pemukiman Terdekat',  '#f39c12'),
]

for i, (col, title, color) in enumerate(cols_jarak):
    axes[i].hist(gdf_koperasi[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    axes[i].axvline(gdf_koperasi[col].mean(), color='black', linestyle='--',
                   label=f'Mean: {gdf_koperasi[col].mean():.0f} m')
    axes[i].set_title(title)
    axes[i].set_xlabel('Jarak (meter)')
    axes[i].set_ylabel('Jumlah Koperasi')
    axes[i].legend()

plt.suptitle('Distribusi Jarak Koperasi ke Fasilitas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_jarak_koperasi.png', bbox_inches='tight')
plt.show()

### 5.3 Korelasi Antar Variabel Ekonomi (Kemiskinan & Pengangguran)

In [ ]:
# Gabungkan kemiskinan dan pengangguran berdasarkan kolom kecamatan
# Sesuaikan nama kolom jika berbeda
if miskin_col and kec_col_mis and tpt_col and kec_col_tpt:
    df_ekonomi = pd.merge(
        kemiskinan[[kec_col_mis, miskin_col]].rename(columns={kec_col_mis: 'kecamatan'}),
        pengangguran[[kec_col_tpt, tpt_col]].rename(columns={kec_col_tpt: 'kecamatan'}),
        on='kecamatan', how='inner'
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter plot kemiskinan vs pengangguran
    axes[0].scatter(df_ekonomi[miskin_col], df_ekonomi[tpt_col],
                   color='#8e44ad', alpha=0.7, s=80, edgecolors='white')
    for _, row in df_ekonomi.iterrows():
        axes[0].annotate(row['kecamatan'], (row[miskin_col], row[tpt_col]),
                        fontsize=6, alpha=0.6)
    axes[0].set_xlabel('Tingkat Kemiskinan (%)')
    axes[0].set_ylabel('Tingkat Pengangguran (%)')
    axes[0].set_title('Kemiskinan vs Pengangguran per Kecamatan')

    # Heatmap korelasi
    corr = df_ekonomi[[miskin_col, tpt_col]].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
               ax=axes[1], square=True, linewidths=1)
    axes[1].set_title('Korelasi Kemiskinan & Pengangguran')

    plt.tight_layout()
    plt.savefig('eda_korelasi_ekonomi.png', bbox_inches='tight')
    plt.show()

    print(f"\nKorelasi Pearson: {df_ekonomi[miskin_col].corr(df_ekonomi[tpt_col]):.3f}")
else:
    print("Kolom kemiskinan/pengangguran belum terdeteksi. Sesuaikan nama kolom di atas.")

### 5.4 Jumlah Fasilitas per Buffer Koperasi (500m & 1km)

In [ ]:
def count_in_buffer(gdf_center, gdf_target, radius_m):
    """Hitung jumlah fitur target dalam radius dari tiap titik center"""
    center_utm = gdf_center.to_crs(epsg=32749).copy()
    target_utm = gdf_target.to_crs(epsg=32749).copy()
    if target_utm.geometry.geom_type.isin(['Polygon','MultiPolygon']).any():
        target_utm['geometry'] = target_utm.geometry.centroid
    buffer = center_utm.copy()
    buffer['geometry'] = buffer.geometry.buffer(radius_m)
    joined = gpd.sjoin(target_utm, buffer[['geometry']], how='inner', predicate='within')
    counts = joined.groupby('index_right').size()
    return center_utm.index.map(counts).fillna(0).astype(int)

print("Menghitung jumlah fasilitas dalam buffer 500m dan 1km...")

gdf_koperasi['faskes_500m']    = count_in_buffer(gdf_koperasi, faskes_pt, 500)
gdf_koperasi['faskes_1km']     = count_in_buffer(gdf_koperasi, faskes_pt, 1000)
gdf_koperasi['pendidikan_500m']= count_in_buffer(gdf_koperasi, pendidikan_pt, 500)
gdf_koperasi['pendidikan_1km'] = count_in_buffer(gdf_koperasi, pendidikan_pt, 1000)

print("\nRata-rata fasilitas dalam buffer:")
print(gdf_koperasi[['faskes_500m','faskes_1km','pendidikan_500m','pendidikan_1km']].mean().round(2))

print("\nKoperasi dengan 0 faskes dalam radius 1km:")
print(gdf_koperasi[gdf_koperasi['faskes_1km']==0][['Nama Koperasi','Kecamatan','Desa']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Faskes dalam buffer
axes[0].bar(['Faskes 500m','Faskes 1km','Pendidikan 500m','Pendidikan 1km'],
            [gdf_koperasi['faskes_500m'].mean(),
             gdf_koperasi['faskes_1km'].mean(),
             gdf_koperasi['pendidikan_500m'].mean(),
             gdf_koperasi['pendidikan_1km'].mean()],
            color=['#e74c3c','#c0392b','#3498db','#2980b9'])
axes[0].set_title('Rata-rata Fasilitas dalam Buffer per Koperasi')
axes[0].set_ylabel('Rata-rata Jumlah Fasilitas')
axes[0].tick_params(axis='x', rotation=20)

# Scatter: faskes 1km vs jarak faskes
axes[1].scatter(gdf_koperasi['jarak_faskes_m'], gdf_koperasi['faskes_1km'],
               alpha=0.6, color='#e74c3c', s=60, edgecolors='white')
axes[1].set_xlabel('Jarak ke Faskes Terdekat (m)')
axes[1].set_ylabel('Jumlah Faskes dalam 1km')
axes[1].set_title('Jarak vs Jumlah Faskes (1km Buffer)')

plt.tight_layout()
plt.savefig('eda_buffer_fasilitas.png', bbox_inches='tight')
plt.show()

### 5.5 Peta Interaktif (Folium)

In [ ]:
# Peta interaktif semua variabel
center_lat = gdf_koperasi['lat'].mean()
center_lon = gdf_koperasi['lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB dark_matter')

# Layer bangunan/pemukiman sebagai heatmap
pemuk_centroids = pemukiman.copy()
if pemuk_centroids.geometry.geom_type.isin(['Polygon','MultiPolygon']).any():
    pemuk_centroids['geometry'] = pemuk_centroids.geometry.centroid
heat_data = [[geom.y, geom.x] for geom in pemuk_centroids.geometry if geom is not None]
HeatMap(heat_data, radius=5, blur=3, min_opacity=0.2, name='Kepadatan Pemukiman').add_to(m)

# Layer faskes
faskes_layer = folium.FeatureGroup(name='Fasilitas Kesehatan')
for _, row in faskes_pt.iterrows():
    name = row.get('name', 'Faskes') if 'name' in row else 'Faskes'
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color='#e74c3c', fill=True, fill_opacity=0.8,
        popup=str(name)
    ).add_to(faskes_layer)
faskes_layer.add_to(m)

# Layer pendidikan
pend_layer = folium.FeatureGroup(name='Fasilitas Pendidikan')
for _, row in pendidikan_pt.iterrows():
    name = row.get('name', 'Sekolah') if 'name' in row else 'Sekolah'
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color='#3498db', fill=True, fill_opacity=0.8,
        popup=str(name)
    ).add_to(pend_layer)
pend_layer.add_to(m)

# Layer koperasi
kop_layer = folium.FeatureGroup(name='Koperasi KDMP')
for _, row in gdf_koperasi.iterrows():
    popup_text = f"""
    <b>{row['Nama Koperasi']}</b><br>
    Kecamatan: {row['Kecamatan']}<br>
    Desa: {row['Desa']}<br>
    Progres: {row['Progres Pembangunan (%)']}%<br>
    Jarak Faskes: {row['jarak_faskes_m']:.0f} m<br>
    Jarak Sekolah: {row['jarak_pendidikan_m']:.0f} m
    """
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(color='green', icon='star', prefix='fa')
    ).add_to(kop_layer)
kop_layer.add_to(m)

folium.LayerControl().add_to(m)

m.save('peta_interaktif_goal2.html')
print("Peta interaktif disimpan: peta_interaktif_goal2.html")
m

### 5.6 Ringkasan Kerentanan Awal per Koperasi

In [ ]:
# Skor kerentanan sederhana berdasarkan jarak ke fasilitas
# Semakin jauh dari fasilitas = semakin rentan
# Normalisasi 0-1 lalu jumlahkan

def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

gdf_koperasi['skor_rentan'] = (
    normalize(gdf_koperasi['jarak_faskes_m']) * 0.30 +
    normalize(gdf_koperasi['jarak_pendidikan_m']) * 0.25 +
    normalize(gdf_koperasi['jarak_jalan_m']) * 0.25 +
    normalize(gdf_koperasi['jarak_pemukiman_m']) * 0.20
)

# Kategorikan
gdf_koperasi['kategori_rentan'] = pd.cut(
    gdf_koperasi['skor_rentan'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Rendah', 'Sedang', 'Tinggi'],
    include_lowest=True
)

print("Distribusi kategori kerentanan:")
print(gdf_koperasi['kategori_rentan'].value_counts())

print("\nTop 10 koperasi dengan kerentanan TINGGI:")
cols_show = ['Nama Koperasi', 'Kecamatan', 'Desa', 'skor_rentan',
             'jarak_faskes_m', 'jarak_jalan_m', 'Progres Pembangunan (%)']
print(gdf_koperasi[gdf_koperasi['kategori_rentan']=='Tinggi']
      .sort_values('skor_rentan', ascending=False)[cols_show].head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribusi skor kerentanan
colors_rentan = {'Rendah': '#2ecc71', 'Sedang': '#f39c12', 'Tinggi': '#e74c3c'}
vc = gdf_koperasi['kategori_rentan'].value_counts()
bars = axes[0].bar(vc.index, vc.values,
                  color=[colors_rentan[k] for k in vc.index])
axes[0].set_title('Distribusi Kategori Kerentanan Koperasi')
axes[0].set_ylabel('Jumlah Koperasi')
for bar, val in zip(bars, vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold')

# Scatter: skor kerentanan vs progres pembangunan
color_map = gdf_koperasi['kategori_rentan'].map(colors_rentan)
axes[1].scatter(gdf_koperasi['skor_rentan'],
               gdf_koperasi['Progres Pembangunan (%)'],
               c=color_map, s=70, alpha=0.8, edgecolors='white')
axes[1].set_xlabel('Skor Kerentanan')
axes[1].set_ylabel('Progres Pembangunan (%)')
axes[1].set_title('Kerentanan vs Progres Pembangunan Koperasi')
patches = [mpatches.Patch(color=v, label=k) for k, v in colors_rentan.items()]
axes[1].legend(handles=patches, title='Kategori Kerentanan')

plt.tight_layout()
plt.savefig('eda_skor_kerentanan.png', bbox_inches='tight')
plt.show()

## 6. Export Hasil EDA

In [ ]:
# Simpan hasil EDA koperasi lengkap
cols_export = [
    'Nama Koperasi', 'Kecamatan', 'Desa', 'lat', 'lon',
    'Progres Pembangunan (%)', 'Kategori Pembangunan',
    'jarak_faskes_m', 'jarak_pendidikan_m', 'jarak_jalan_m', 'jarak_pemukiman_m',
    'faskes_500m', 'faskes_1km', 'pendidikan_500m', 'pendidikan_1km',
    'skor_rentan', 'kategori_rentan'
]
df_export = gdf_koperasi[cols_export].copy()
df_export.to_csv('hasil_eda_koperasi_goal2.csv', index=False)

print("File tersimpan:")
print("  hasil_eda_koperasi_goal2.csv")
print("  peta_interaktif_goal2.html")
print("  peta_semua_variabel.png")
print("  eda_jarak_koperasi.png")
print("  eda_skor_kerentanan.png")
print("  (dan file png lainnya)")
print(f"\nTotal koperasi dianalisis: {len(df_export)}")
print(df_export['kategori_rentan'].value_counts())